# 04 · Generation and comparisons

Run MODE='base' before training for baseline outputs. After training, attach checkpoint outputs and run MODE='trained'. Generations persist after each prompt.

Upload this notebook to Kaggle. Enable Internet; select **GPU T4 x2** for training/generation. The judge notebook needs benchmark model access, not a GPU. Enter your public GitHub URL below. No GitHub credentials are needed.


## 1. Configuration

Run All defaults to smoke checks only. After inspecting them, set `RUN_FULL=True`. Use a new run name if changing experimental settings; completed runs are skipped.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = "https://github.com/0x0shephard/genai-pa2.git"
GIT_REF = "main"  # After the first run, use its printed commit SHA for all final runs.
REPO_DIR = Path("/kaggle/working/pa2-repo")
OUTPUT_ROOT = Path("/kaggle/working/outputs")
PREPARED_ROOT = Path("/kaggle/working/prepared")
RUN_SMOKE = True
RUN_FULL = False  # Inspect smoke output, then set True and rerun the configuration/full cells.

MODE = "base"  # "base" or "trained"
SFT_BEST = Path("/kaggle/input/YOUR-SFT-OUTPUT/outputs/sft/best")
HH_BEST = Path("/kaggle/input/YOUR-HH-OUTPUT/outputs/dpo_hh/best")
UF_BEST = Path("/kaggle/input/YOUR-UF-OUTPUT/outputs/dpo_ultrafeedback/best")
PERSONA_BEST = Path("/kaggle/input/YOUR-PERSONA-OUTPUT/outputs/persona/best")
# In trained mode, attach SFT prepared prompts to preserve the exact Alpaca subset.
ALPACA_PROMPTS = None  # e.g. /kaggle/input/YOUR-SFT-OUTPUT/prepared/eval/alpaca_eval_300_seed42.json
GENERATION_RESUME_DIR = None  # A saved output directory containing generation JSONs and manifests.
MAX_NEW_TOKENS = 256
REPETITION_PENALTY = 1.0
NO_REPEAT_NGRAM_SIZE = 0


## 2. Clone the chosen GitHub revision and install dependencies


In [ ]:
assert "YOUR_" not in REPO_URL, "Enter your public GitHub repository URL in the configuration cell."
assert REPO_URL.startswith("https://github.com/"), "Use a public GitHub HTTPS URL."
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    assert origin.rstrip("/").removesuffix(".git") == REPO_URL.rstrip("/").removesuffix(".git"), "Existing checkout belongs to another repository."
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True).strip(), "Checkout has local changes; preserve them before changing revisions."
subprocess.check_call(["git", "fetch", "origin", GIT_REF], cwd=REPO_DIR)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
os.environ["ALIGNMENT_GIT_REF"] = GIT_REF
os.environ["PYTHONPATH"] = str(REPO_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/tmp/pa2-hf-cache"  # Caches are disposable, not saved notebook artifacts.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
print("Git commit:", COMMIT)

def run(*args):
    subprocess.check_call([sys.executable, "-m", "alignment", *map(str, args)], cwd=REPO_DIR)

def need(path, label):
    path = Path(path)
    if not path.exists():
        print("Available attachments:", list(Path("/kaggle/input").glob("*/*")))
        raise FileNotFoundError(f"Attach the saved {label} output and update its configuration path: {path}")
    return path

def completed(path):
    summary = Path(path) / "summary.json"
    return summary.exists() and json.loads(summary.read_text()).get("completed", False)

def show_samples(path, count=5):
    for row in json.loads(Path(path).read_text())[:count]:
        print("PROMPT:", row["instruction"])
        print("RESPONSE:", row["output"], "\n")


In [ ]:
# Keep Kaggle's installed CUDA PyTorch. These requirements do not pin torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-persona.txt"), "--upgrade-strategy", "only-if-needed"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"])
subprocess.check_call(["nvidia-smi"])
subprocess.check_call([sys.executable, "-c", "import torch, transformers, datasets; print('torch', torch.__version__, 'transformers', transformers.__version__, 'datasets', datasets.__version__); print('GPU count:', torch.cuda.device_count()); assert torch.cuda.device_count() >= 2, 'Enable GPU T4 x2'"])
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"], cwd=REPO_DIR)


## 3. Fix evaluation prompts and model jobs


In [ ]:
assert MODE in ("base", "trained")
if ALPACA_PROMPTS:
    alpaca = need(ALPACA_PROMPTS, "fixed AlpacaEval prompts")
else:
    run("prepare", "--track", "alpaca", "--output", PREPARED_ROOT / "eval")
    alpaca = PREPARED_ROOT / "eval/alpaca_eval_300_seed42.json"
hh = REPO_DIR / "eval_sets/hh_rlhf_eval_300_seed42.json"
persona_prompts = REPO_DIR / "eval_sets/persona_eval_40_seed42.json"
jobs = [("qwen2.5-0.5b-base", "Qwen/Qwen2.5-0.5B", alpaca, False)] if MODE == "base" else [
    ("sft_alpaca", SFT_BEST, alpaca, False), ("dpo_ultrafeedback", UF_BEST, alpaca, False),
    ("sft_hh", SFT_BEST, hh, False), ("dpo_hh", HH_BEST, hh, False),
    ("smollm2-base", "HuggingFaceTB/SmolLM2-360M-Instruct", persona_prompts, True),
    ("persona_lora", PERSONA_BEST, persona_prompts, True)]
for _, checkpoint, _, _ in jobs:
    if isinstance(checkpoint, Path):
        need(checkpoint / "alignment_manifest.json", "trained checkpoint")

def generate_job(name, checkpoint, prompts, is_persona, smoke=False):
    folder = OUTPUT_ROOT / ("sanity" if smoke else "generations")
    args = ["generate", "--checkpoint", checkpoint, "--prompts", prompts,
            "--output", folder / f"{name}.json", "--generator", name,
            "--max-new-tokens", MAX_NEW_TOKENS, "--repetition-penalty", REPETITION_PENALTY,
            "--no-repeat-ngram-size", NO_REPEAT_NGRAM_SIZE]
    if is_persona:
        args += ["--persona"]
    if smoke:
        args += ["--limit", 5]
    elif GENERATION_RESUME_DIR and (Path(GENERATION_RESUME_DIR) / f"{name}.json").exists():
        args += ["--resume-from", Path(GENERATION_RESUME_DIR) / f"{name}.json"]
    run(*args)
    return folder / f"{name}.json"


## 4. Inspect five responses per model

Confirm paths, prompt formatting, EOS, length, and repetition before launching the full sets. If decoding changes are needed, use a new output directory and apply them to both sides.


In [ ]:
if RUN_SMOKE:
    for job in jobs:
        show_samples(generate_job(*job, smoke=True))


## 5. Full generation and blind comparison files


In [ ]:
if RUN_FULL:
    for job in jobs:
        generate_job(*job)
    if MODE == "trained":
        generations = OUTPUT_ROOT / "generations"
        for track, baseline, candidate in [("hh", "sft_hh", "dpo_hh"),
            ("ultrafeedback", "sft_alpaca", "dpo_ultrafeedback"),
            ("persona", "smollm2-base", "persona_lora")]:
            run("pairs", "--baseline", generations / f"{baseline}.json",
                "--candidate", generations / f"{candidate}.json",
                "--output", OUTPUT_ROOT / "pairs" / f"{track}.jsonl", "--track", track)
else:
    print("Inspect the sanity responses, then set RUN_FULL=True.")


## Preserve outputs

Use **Save Version → Save & Run All** after setting `RUN_FULL=True`, or save completed interactive outputs with the appropriate Kaggle version option. Attach that saved notebook's output to the next stage using **Add Input → Notebook Output**. Confirm the files exist in the saved version before ending the session.

New artifacts are under `/kaggle/working/outputs` and `/kaggle/working/prepared`. Model files are large: do not duplicate them into zip archives unless disk permits. For interruption recovery, attach the prior saved output and configure its resume path. A file in a temporary session is preserved only when Kaggle saves the output.
